# COFFEEBEAN — ANC Training on Google Colab
**Phase 5** — GPU-accelerated training with MLflow tracking to DagsHub

## ⚠️ Setup Required (one-time)
Before running this notebook, you need:
1. A **DagsHub** account with the `COFFEEBEAN` repo connected → [dagshub.com](https://dagshub.com)
2. Your **DagsHub token** (Settings → Tokens)
3. **Google Drive** mounted (for DVC dataset)

See `docs/colab_setup.md` for step-by-step instructions.

In [ ]:
# ── Parameters (injected by Airflow via papermill) ─────────────────────────
DAGSHUB_USERNAME = ""         # ⚠️ ACCOUNT REQUIRED: set your DagsHub username
DAGSHUB_TOKEN    = ""         # ⚠️ ACCOUNT REQUIRED: set your DagsHub token
REPO_NAME        = "COFFEEBEAN"
GITHUB_REPO      = "Zenith1415/COFFEEBEAN"
EPOCHS           = 50
BATCH_SIZE       = 32
LEARNING_RATE    = 0.001
COLAB_DRIVE_PATH = "/content/drive/MyDrive/COFFEEBEAN"

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install mlflow dvc dvc-gdrive torchmetrics pystoi onnx onnxruntime pyyaml -q

In [ ]:
# ── 2. Mount Google Drive (for dataset) ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# ── 3. Clone repo and configure MLflow → DagsHub ──────────────────────────
import os

# ⚠️ ACCOUNT REQUIRED: set these before running
if not DAGSHUB_USERNAME or not DAGSHUB_TOKEN:
    raise ValueError(
        'Set DAGSHUB_USERNAME and DAGSHUB_TOKEN in the Parameters cell.\n'
        'Get your token at: https://dagshub.com/user/settings/tokens'
    )

os.environ['MLFLOW_TRACKING_URI']   = f'https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow'
os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN
os.environ['PYTHONUTF8']            = '1'

!git clone https://github.com/{GITHUB_REPO}.git /content/COFFEEBEAN
%cd /content/COFFEEBEAN
print('Repo cloned and MLflow configured')

In [ ]:
# ── 4. Pull dataset from Google Drive via DVC ─────────────────────────────
# ⚠️ ACCOUNT REQUIRED: DVC Google Drive remote must be configured
# See: docs/colab_setup.md

# Fallback: copy from mounted Drive directly
import shutil
from pathlib import Path

drive_data = Path(COLAB_DRIVE_PATH) / 'data'
if drive_data.exists():
    shutil.copytree(str(drive_data), 'data', dirs_exist_ok=True)
    print(f'Dataset copied from Drive: {list(Path("data/raw").rglob("*.wav")).__len__()} WAV files')
else:
    print(f'Drive path not found: {drive_data}')
    print('Running DVC pull...')
    !dvc pull

In [ ]:
# ── 5. Check GPU ───────────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 6. Override config for Colab session ──────────────────────────────────
import yaml
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['training']['epochs']         = EPOCHS
cfg['training']['batch_size']     = BATCH_SIZE
cfg['training']['learning_rate']  = LEARNING_RATE
with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('Config updated:', cfg['training'])

In [ ]:
# ── 7. Train ───────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/COFFEEBEAN')
from src.training.train import train

run_id = train()
print(f'Training complete. Run ID: {run_id}')

In [ ]:
# ── 8. Evaluate ────────────────────────────────────────────────────────────
from src.evaluation.run_evaluation import run_evaluation

results = run_evaluation(
    checkpoint='models/anc_checkpoint.pt',
    run_id=run_id,
    snr_levels=[-5.0, 0.0, 5.0, 10.0],
    num_pairs=10,
)
print('Evaluation complete')
for tag, metrics in results.items():
    print(f"  {tag}: SNR improvement={metrics.get('snr_improvement')} dB")

In [ ]:
# ── 9. Export ONNX ────────────────────────────────────────────────────────
from src.deployment.export import export_to_onnx, benchmark_onnx
from src.training.model import ANCAudioModel
import torch

model = ANCAudioModel(cfg)
model.load_state_dict(torch.load('models/anc_checkpoint.pt', map_location='cpu'))

export_info = export_to_onnx(model, 'models/anc_model.onnx', sample_rate=16000)
bench       = benchmark_onnx('models/anc_model.onnx', sample_rate=16000)

print('ONNX Export:', export_info)
print('Benchmark:',   bench)

In [ ]:
# ── 10. Save artifacts back to Drive ──────────────────────────────────────
import shutil
from pathlib import Path

dest = Path(COLAB_DRIVE_PATH) / 'models'
dest.mkdir(parents=True, exist_ok=True)

for f in ['models/anc_checkpoint.pt', 'models/anc_model.onnx']:
    if Path(f).exists():
        shutil.copy(f, str(dest / Path(f).name))
        print(f'Saved {f} -> Drive')

print(f'All artifacts saved to: {dest}')